In [9]:
import os
import json
import pandas as pd
import traceback

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [32]:
!pip install langchain_groq


   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 2/2 [langchain_groq]



In [38]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load variables from .env file
load_dotenv(r"C:\Users\munee\mcqgen\.env") # This reads .env and sets os.environ automatically

# Optional: Check if key is loaded (for debugging)
if "GOOGLE_API_KEY" not in os.environ:
    print("Warning: GOOGLE_API_KEY not found in .env file!")
else:
    print("Google API key loaded successfully.")

# Now create the LLM — it will use the env var automatically
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

Google API key loaded successfully.


In [39]:
response = llm.invoke("Can you give me the result of today's cricket match?")
print(response.content)

I'm not aware of the current date or any cricket matches that may be happening today. I'm a large language model, my training data only goes up until 2023, and I don't have real-time access to information. However, I can suggest some ways for you to find the result of today's cricket match:

1. Check online cricket websites: You can check websites like ESPN Cricinfo, Cricket Australia, or the International Cricket Council (ICC) website for the latest cricket news and scores.
2. Follow cricket social media: Follow cricket teams, players, and commentators on social media platforms like Twitter, Instagram, or Facebook to get updates on the match.
3. Watch live streaming: If the match is being broadcast live, you can watch it on TV or online streaming platforms like Hotstar, Willow TV, or Sky Sports.
4. Check sports news websites: Websites like BBC Sport, Sky Sports, or Fox Sports often provide live updates and scores of cricket matches.

Please note that the result of today's cricket matc

In [40]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='The translation of "I love programming" to French is:\n\n"J\'adore le programmation."', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 55, 'total_tokens': 77, 'completion_time': 0.036389973, 'completion_tokens_details': None, 'prompt_time': 0.003030078, 'prompt_tokens_details': None, 'queue_time': 0.005107749, 'total_time': 0.039420051}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c71e3-2c41-7fa0-9d21-6260db6aa542-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 22, 'total_tokens': 77})

In [42]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence
import PyPDF2

In [48]:
RESPONSE_JSON = {
    "1":{
        "mcq":"multiple choice question",
        "options":{
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct":"correct answer",
    },
        "2":{
        "mcq":"multiple choice question",
        "options":{
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct":"correct answer",

    },
        "3":{
        "mcq":"multiple choice question",
        "options":{
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct":"correct answer",

    },

}

In [49]:
Template="""
Text:{text}
You are an expert MCQ maker. Given the above test, it is your job \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [50]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone","response_json"],
    template=Template
)

In [57]:
# Create the chain (replaces LLMChain)
quiz_chain = quiz_generation_prompt | llm | StrOutputParser()

# Run the chain (replaces chain.run())
response = quiz_chain.invoke({
    "text": "I am a student",
    "number": 4,
    "subject": "Generative AI",
    "tone": "simple",
    "response_json": json.dumps(RESPONSE_JSON, indent=2)
})

print(response)

{
  "1": {
    "mcq": "What is the primary function of a Generative AI model?",
    "options": {
      "a": "To classify and categorize data",
      "b": "To generate new, original content",
      "c": "To analyze and predict trends",
      "d": "To translate languages"
    },
    "correct": "b"
  },
  "2": {
    "mcq": "What is the term for the process of training a Generative AI model on a dataset?",
    "options": {
      "a": "Inference",
      "b": "Fine-tuning",
      "c": "Pre-training",
      "d": "Data augmentation"
    },
    "correct": "c"
  },
  "3": {
    "mcq": "What is a common application of Generative AI in the field of art?",
    "options": {
      "a": "Creating realistic portraits",
      "b": "Generating music and audio tracks",
      "c": "Designing and creating 3D models",
      "d": All of the above"
    },
    "correct": "d"
  },
  "4": {
    "mcq": "What is the term for the ability of a Generative AI model to produce diverse and varied outputs?",
    "options"

In [58]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.
You need to evaluate the complexity of the question and give a complete json analysis of the quiz. Only use max 50 words for each 
if the quiz is evaluate not at per with the complexity of the cognitive and analytical abilities of the students,
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the students' 
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [59]:
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2
)

In [60]:
# Create the chain (replaces LLMChain)
review_chain = quiz_evaluation_prompt | llm | StrOutputParser()
# Run the chain (replaces chain.run())
response = review_chain.invoke({
    "subject": "Generative AI",
    "quiz": "simple",
    "response_json": json.dumps(RESPONSE_JSON, indent=2)
})

print(response)

**Quiz Analysis**

The quiz appears to be simple, but upon closer inspection, it lacks depth and complexity suitable for Generative AI students. Here's a detailed analysis:

**Quiz Complexity:** 2/10 (Too basic for AI students)

**Quiz Type:** Multiple Choice Questions (MCQs)

**Quiz Format:** 5 questions, each with 4 options

**Quiz Content:** Grammar and sentence structure

**Quiz Difficulty:** Easy

**Quiz Analysis JSON:**
```json
{
  "quiz_complexity": 2,
  "quiz_type": "MCQs",
  "quiz_format": "5 questions, 4 options each",
  "quiz_content": "Grammar and sentence structure",
  "quiz_difficulty": "Easy"
}
```

**Recommendations:**

1. **Increase complexity**: Add more nuanced questions that require critical thinking and analysis.
2. **Add context**: Provide more context to each question to make it more relevant and challenging.
3. **Use more advanced grammar concepts**: Incorporate more complex grammar concepts, such as clause structure, verb tenses, and sentence types.
4. **Use AI

In [61]:
generate_evaluate_chain = (
    quiz_generation_prompt 
    | llm 
    | StrOutputParser() 
    | (lambda quiz: {"subject": "your_subject", "quiz": quiz})
    | quiz_evaluation_prompt 
    | llm 
    | StrOutputParser()
)

In [63]:
quiz = quiz_chain.invoke({
    "text": "I am a student",
    "number": 4,
    "subject": "Biology",
    "tone": "simple",
    "response_json": json.dumps(RESPONSE_JSON, indent=2)
})

In [64]:
review = review_chain.invoke({
    "subject": "Biology",
    "quiz": quiz
})

In [65]:

print("QUIZ:", quiz)
print("REVIEW:", review)

QUIZ: {
  "1": {
    "mcq": "What is the process by which plants make their own food?",
    "options": {
      "a": "Respiration",
      "b": "Photosynthesis",
      "c": "Decomposition",
      "d": "Fermentation"
    },
    "correct": "b"
  },
  "2": {
    "mcq": "Which part of a plant cell is responsible for storing water and nutrients?",
    "options": {
      "a": "Mitochondria",
      "b": "Chloroplast",
      "c": "Vacuole",
      "d": "Nucleus"
    },
    "correct": "c"
  },
  "3": {
    "mcq": "What is the scientific term for the 'building blocks of life'?",
    "options": {
      "a": "Cells",
      "b": "Molecules",
      "c": "Tissues",
      "d": "Organs"
    },
    "correct": "a"
  },
  "4": {
    "mcq": "Which type of organism is characterized by the presence of a backbone?",
    "options": {
      "a": "Mammals",
      "b": "Birds",
      "c": "Reptiles",
      "d": "All of the above"
    },
    "correct": "d"
  }
}
REVIEW: **Quiz Analysis**

The provided quiz consists o

In [67]:
file_path = r"C:\Users\munee\mcqgen\data.txt"

In [70]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [72]:
print(TEXT)

The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]

The earliest machine learning program was introduced in the 1950s when Arthur Samuel invented a computer program that calculated the winning chance in checkers for each side, but the history of machine learning roots back to decades of human desire and effort to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] Hebb's model of neurons interacting with one another set a groundwork for how AIs and machine learning algorithms work under nodes, or artificial neurons used by computers to communicate data.[9] Other researchers who have studied human cognitive systems contri

In [73]:
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [77]:
NUMBER = 5,
SUBJECT = "Machine Learning",
TONE = "simple"

In [78]:
# No need for get_openai_callback since we're using Groq (it's free)
# Just invoke the chains directly

# Run quiz generation chain
quiz = quiz_chain.invoke({
    "text": TEXT,
    "number": NUMBER,
    "subject": SUBJECT,
    "tone": TONE,
    "response_json": json.dumps(RESPONSE_JSON)
})

# Run review chain with quiz output
review = review_chain.invoke({
    "subject": SUBJECT,
    "quiz": quiz
})

print("QUIZ:\n", quiz)
print("\nREVIEW:\n", review)

QUIZ:
 ### RESPONSE_JSON
{"1": {"mcq": "Who coined the term 'machine learning' in 1959?", "options": {"a": "Arthur Samuel", "b": "Donald Hebb", "c": "Walter Pitts", "d": "Nils Nilsson"}, "correct": "a"}, 
"2": {"mcq": "What was the main objective of the Cybertron machine developed by Raytheon Company in the early 1960s?", "options": {"a": "To recognize patterns in speech", "b": "To analyze sonar signals and electrocardiograms", "c": "To classify images", "d": "To translate languages"}, "correct": "b"}, 
"3": {"mcq": "What is the main difference between supervised and unsupervised learning algorithms?", "options": {"a": "Supervised learning requires labeled data, while unsupervised learning does not", "b": "Supervised learning is used for classification, while unsupervised learning is used for regression", "c": "Supervised learning is used for clustering, while unsupervised learning is used for dimensionality reduction", "d": "Supervised learning is used for reinforcement learning, whil

In [79]:
response

'**Quiz Analysis**\n\nThe quiz appears to be simple, but upon closer inspection, it lacks depth and complexity suitable for Generative AI students. Here\'s a detailed analysis:\n\n**Quiz Complexity:** 2/10 (Too basic for AI students)\n\n**Quiz Type:** Multiple Choice Questions (MCQs)\n\n**Quiz Format:** 5 questions, each with 4 options\n\n**Quiz Content:** Grammar and sentence structure\n\n**Quiz Difficulty:** Easy\n\n**Quiz Analysis JSON:**\n```json\n{\n  "quiz_complexity": 2,\n  "quiz_type": "MCQs",\n  "quiz_format": "5 questions, 4 options each",\n  "quiz_content": "Grammar and sentence structure",\n  "quiz_difficulty": "Easy"\n}\n```\n\n**Recommendations:**\n\n1. **Increase complexity**: Add more nuanced questions that require critical thinking and analysis.\n2. **Add context**: Provide more context to each question to make it more relevant and challenging.\n3. **Use more advanced grammar concepts**: Incorporate more complex grammar concepts, such as clause structure, verb tenses, 

In [82]:
print(repr(quiz))

'### RESPONSE_JSON\n{"1": {"mcq": "Who coined the term \'machine learning\' in 1959?", "options": {"a": "Arthur Samuel", "b": "Donald Hebb", "c": "Walter Pitts", "d": "Nils Nilsson"}, "correct": "a"}, \n"2": {"mcq": "What was the main objective of the Cybertron machine developed by Raytheon Company in the early 1960s?", "options": {"a": "To recognize patterns in speech", "b": "To analyze sonar signals and electrocardiograms", "c": "To classify images", "d": "To translate languages"}, "correct": "b"}, \n"3": {"mcq": "What is the main difference between supervised and unsupervised learning algorithms?", "options": {"a": "Supervised learning requires labeled data, while unsupervised learning does not", "b": "Supervised learning is used for classification, while unsupervised learning is used for regression", "c": "Supervised learning is used for clustering, while unsupervised learning is used for dimensionality reduction", "d": "Supervised learning is used for reinforcement learning, while

In [85]:
import re

json_match = re.search(r'\{.*\}', quiz, re.DOTALL)
if json_match:
    quiz_json = json.loads(json_match.group())
    print(quiz_json)
else:
    print("No JSON found")

{'1': {'mcq': "Who coined the term 'machine learning' in 1959?", 'options': {'a': 'Arthur Samuel', 'b': 'Donald Hebb', 'c': 'Walter Pitts', 'd': 'Nils Nilsson'}, 'correct': 'a'}, '2': {'mcq': 'What was the main objective of the Cybertron machine developed by Raytheon Company in the early 1960s?', 'options': {'a': 'To recognize patterns in speech', 'b': 'To analyze sonar signals and electrocardiograms', 'c': 'To classify images', 'd': 'To translate languages'}, 'correct': 'b'}, '3': {'mcq': 'What is the main difference between supervised and unsupervised learning algorithms?', 'options': {'a': 'Supervised learning requires labeled data, while unsupervised learning does not', 'b': 'Supervised learning is used for classification, while unsupervised learning is used for regression', 'c': 'Supervised learning is used for clustering, while unsupervised learning is used for dimensionality reduction', 'd': 'Supervised learning is used for reinforcement learning, while unsupervised learning is 

In [86]:
print(quiz_json)

{'1': {'mcq': "Who coined the term 'machine learning' in 1959?", 'options': {'a': 'Arthur Samuel', 'b': 'Donald Hebb', 'c': 'Walter Pitts', 'd': 'Nils Nilsson'}, 'correct': 'a'}, '2': {'mcq': 'What was the main objective of the Cybertron machine developed by Raytheon Company in the early 1960s?', 'options': {'a': 'To recognize patterns in speech', 'b': 'To analyze sonar signals and electrocardiograms', 'c': 'To classify images', 'd': 'To translate languages'}, 'correct': 'b'}, '3': {'mcq': 'What is the main difference between supervised and unsupervised learning algorithms?', 'options': {'a': 'Supervised learning requires labeled data, while unsupervised learning does not', 'b': 'Supervised learning is used for classification, while unsupervised learning is used for regression', 'c': 'Supervised learning is used for clustering, while unsupervised learning is used for dimensionality reduction', 'd': 'Supervised learning is used for reinforcement learning, while unsupervised learning is 

In [89]:
import json
print(json.dumps(quiz_json, indent=2))

{
  "1": {
    "mcq": "Who coined the term 'machine learning' in 1959?",
    "options": {
      "a": "Arthur Samuel",
      "b": "Donald Hebb",
      "c": "Walter Pitts",
      "d": "Nils Nilsson"
    },
    "correct": "a"
  },
  "2": {
    "mcq": "What was the main objective of the Cybertron machine developed by Raytheon Company in the early 1960s?",
    "options": {
      "a": "To recognize patterns in speech",
      "b": "To analyze sonar signals and electrocardiograms",
      "c": "To classify images",
      "d": "To translate languages"
    },
    "correct": "b"
  },
  "3": {
    "mcq": "What is the main difference between supervised and unsupervised learning algorithms?",
    "options": {
      "a": "Supervised learning requires labeled data, while unsupervised learning does not",
      "b": "Supervised learning is used for classification, while unsupervised learning is used for regression",
      "c": "Supervised learning is used for clustering, while unsupervised learning is us

In [91]:
quiz_table_data = []
for key, value in quiz_json.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}:{option_value}"
            for option, option_value in value["options"].items()
        ]
    )
    correct = value["correct"]
    quiz_table_data.append({"MCQ":mcq, "Choices":options, "Correct":correct})

In [92]:
quiz_table_data

[{'MCQ': "Who coined the term 'machine learning' in 1959?",
  'Choices': 'a:Arthur Samuel | b:Donald Hebb | c:Walter Pitts | d:Nils Nilsson',
  'Correct': 'a'},
 {'MCQ': 'What was the main objective of the Cybertron machine developed by Raytheon Company in the early 1960s?',
  'Choices': 'a:To recognize patterns in speech | b:To analyze sonar signals and electrocardiograms | c:To classify images | d:To translate languages',
  'Correct': 'b'},
 {'MCQ': 'What is the main difference between supervised and unsupervised learning algorithms?',
  'Choices': 'a:Supervised learning requires labeled data, while unsupervised learning does not | b:Supervised learning is used for classification, while unsupervised learning is used for regression | c:Supervised learning is used for clustering, while unsupervised learning is used for dimensionality reduction | d:Supervised learning is used for reinforcement learning, while unsupervised learning is used for decision-making',
  'Correct': 'a'},
 {'MCQ'

In [94]:
quiz = pd.DataFrame(quiz_table_data)

In [95]:
quiz.to_csv("machinelearning.csv", index=False)